In [1]:
! pip install autograd

In [2]:
import autograd.numpy as np  # autograd専用のnumpyをインポート
from autograd import grad, hessian

def newton_with_autograd(f, x0, tol=1e-6, max_iter=50):
    # 関数fから勾配関数とヘッセ行列関数を自動生成
    f_grad = grad(f)
    f_hessian = hessian(f)

    xk = x0.astype(float)

    for k in range(max_iter):
        g = f_grad(xk)
        H = f_hessian(xk)

        # 終了判定
        if np.linalg.norm(g) < tol:
            print(f"Converged in {k} iterations.")
            return xk

        # ニュートン方程式 H * d = -g を解く
        try:
            d = np.linalg.solve(H, -g)
        except np.linalg.linalg.LinAlgError:
            print("Hessian is singular. Convergence failed.")
            return None

        xk = xk + d
        print(f"Iter {k+1}: x = {xk}, f(x) = {f(xk):.6f}")

    return xk

# --- 使用例: 複雑な非線形関数 ---
# f(x, y) = (x - 1)^2 + (y - 2.5)^2 + exp(x - y)
def objective_function(x):
    return (x[0] - 1.0)**2 + (x[1] - 2.5)**2 + np.exp(x[0] - x[1])

x_start = np.array([0.0, 0.0])
result = newton_with_autograd(objective_function, x_start)
print(f"\nFinal Result: {result}")

Iter 1: x = [1.125 2.375], f(x) = 0.317755
Iter 2: x = [0.91648745 2.58351255], f(x) = 0.202757
Iter 3: x = [0.90732583 2.59267417], f(x) = 0.202557
Iter 4: x = [0.90731254 2.59268746], f(x) = 0.202557
Converged in 4 iterations.

Final Result: [0.90731254 2.59268746]


In [3]:
import numpy as np
from autograd import grad

class LBFGS:
    def __init__(self, m=10):
        self.m = m  # 記憶する履歴の数
        self.history = []  # (s_k, y_k, rho_k) のリスト

    def two_loop_recursion(self, g):
        q = g.copy()
        alphas = []

        # Backward pass
        for s, y, rho in reversed(self.history):
            alpha = rho * np.dot(s, q)
            q = q - alpha * y
            alphas.append(alpha)

        # 初期ヘッセ行列の近似（スケーリング）
        if len(self.history) > 0:
            s_last, y_last, rho_last = self.history[-1]
            gamma = np.dot(s_last, y_last) / np.dot(y_last, y_last)
        else:
            gamma = 1.0

        r = gamma * q

        # Forward pass
        for (s, y, rho), alpha in zip(self.history, reversed(alphas)):
            beta = rho * np.dot(y, r)
            r = r + s * (alpha - beta)

        return -r  # 探索方向 d

    def solve(self, f, x0, tol=1e-5, max_iter=100):
        f_grad = grad(f)
        x = x0.astype(float)
        g = f_grad(x)

        for k in range(max_iter):
            if np.linalg.norm(g) < tol:
                print(f"Converged at iter {k}")
                return x

            # 1. 探索方向の決定
            d = self.two_loop_recursion(g)

            # 2. 本来はここで「ラインサーチ」を行うが、簡略化のため固定値または簡易計算
            alpha = 0.1  # 本来はWolfe条件を満たすalphaを探す

            x_next = x + alpha * d
            g_next = f_grad(x_next)

            # 3. 履歴の更新
            s = x_next - x
            y = g_next - g
            rho_inv = np.dot(y, s)

            if rho_inv > 1e-10:  # 数値的安定性のためのチェック
                rho = 1.0 / rho_inv
                self.history.append((s, y, rho))
                if len(self.history) > self.m:
                    self.history.pop(0)

            x, g = x_next, g_next
            print(f"Iter {k}: f(x) = {f(x):.6f}")

        return x

# --- テスト実行 ---
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

optimizer = LBFGS(m=5)
start_pos = np.array([-1.2, 1.0])
result = optimizer.solve(rosenbrock, start_pos)
print("Result:", result)

Iter 0: f(x) = 16380979.721216
Iter 1: f(x) = 10399088.646995
Iter 2: f(x) = 9218048.150646
Iter 3: f(x) = 8079999.241324
Iter 4: f(x) = 7084770.704815
Iter 5: f(x) = 6211683.022497
Iter 6: f(x) = 5445839.782007
Iter 7: f(x) = 4774077.197882
Iter 8: f(x) = 4184850.612663
Iter 9: f(x) = 3668031.858199
Iter 10: f(x) = 3214735.567738
Iter 11: f(x) = 2817166.621677
Iter 12: f(x) = 2468486.327776
Iter 13: f(x) = 2162695.034014
Iter 14: f(x) = 1894529.159137
Iter 15: f(x) = 1659370.863875
Iter 16: f(x) = 1453168.811130
Iter 17: f(x) = 1272368.653376
Iter 18: f(x) = 1113852.052779
Iter 19: f(x) = 974883.186236
Iter 20: f(x) = 853061.816271
Iter 21: f(x) = 746282.121609
Iter 22: f(x) = 652696.580247
Iter 23: f(x) = 570684.284718
Iter 24: f(x) = 498823.145399
Iter 25: f(x) = 435865.504556
Iter 26: f(x) = 380716.742432
Iter 27: f(x) = 332416.508092
Iter 28: f(x) = 290122.252855
Iter 29: f(x) = 253094.783688
Iter 30: f(x) = 220685.588645
Iter 31: f(x) = 192325.716886
Iter 32: f(x) = 167516.022488

In [8]:
import numpy as np
from autograd import grad

def line_search(f, f_grad, x, d, alpha=1.0, rho=0.5, c=1e-4):
    """
    簡易版ラインサーチ (Armijo条件のみ)
    :param c: 減少の定数 (通常は非常に小さい値)
    :param rho: 歩幅を小さくする割合
    """
    f_x = f(x)
    g_x = f_grad(x)

    # Armijo条件: f(x + alpha*d) <= f(x) + c * alpha * (g^T * d)
    while f(x + alpha * d) > f_x + c * alpha * np.dot(g_x, d):
        alpha *= rho
        if alpha < 1e-10: # ループ脱出用
            break
    return alpha

class LBFGS_Advanced:
    def __init__(self, m=10):
        self.m = m
        self.history = []

    def solve(self, f, x0, tol=1e-5, max_iter=100):
        f_grad = grad(f)
        x = x0.astype(float)

        for k in range(max_iter):
            g = f_grad(x)
            if np.linalg.norm(g) < tol:
                return x

            # 1. 探索方向を決定 (Two-loop recursion)
            d = self.two_loop_recursion(g)

            # 2. ラインサーチで最適な alpha を決定
            alpha = line_search(f, f_grad, x, d)

            # 3. 更新
            x_next = x + alpha * d
            g_next = f_grad(x_next)

            # 4. 履歴の保存
            s = x_next - x
            y = g_next - g
            sy = np.dot(y, s)
            if sy > 1e-10:
                self.history.append((s, y, 1.0/sy))
                if len(self.history) > self.m:
                    self.history.pop(0)

            x = x_next
            print(f"Iter {k}: f(x) = {f(x):.8f}, alpha = {alpha:.4f}")
        return x

    def two_loop_recursion(self, g):
        q = g.copy()
        alphas = []

        # Backward pass
        for s, y, rho in reversed(self.history):
            alpha = rho * np.dot(s, q)
            q = q - alpha * y
            alphas.append(alpha)

        # 初期ヘッセ行列の近似（スケーリング）
        if len(self.history) > 0:
            s_last, y_last, rho_last = self.history[-1]
            gamma = np.dot(s_last, y_last) / np.dot(y_last, y_last)
        else:
            gamma = 1.0

        r = gamma * q

        # Forward pass
        for (s, y, rho), alpha in zip(self.history, reversed(alphas)):
            beta = rho * np.dot(y, r)
            r = r + s * (alpha - beta)

        return -r  # 探索方向 d

# --- テスト実行 ---
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

optimizer = LBFGS_Advanced()
start_pos = np.array([-1.2, 1.0])
result = optimizer.solve(rosenbrock, start_pos)
print("Result:", result)



Iter 0: f(x) = 5.10111266, alpha = 0.0010
Iter 1: f(x) = 4.15378843, alpha = 1.0000
Iter 2: f(x) = 4.11721504, alpha = 1.0000
Iter 3: f(x) = 4.11401723, alpha = 1.0000
Iter 4: f(x) = 4.11081850, alpha = 1.0000
Iter 5: f(x) = 4.10761875, alpha = 1.0000
Iter 6: f(x) = 4.10441796, alpha = 1.0000
Iter 7: f(x) = 4.10121613, alpha = 1.0000
Iter 8: f(x) = 4.09801326, alpha = 1.0000
Iter 9: f(x) = 4.09480936, alpha = 1.0000
Iter 10: f(x) = 4.09160442, alpha = 1.0000
Iter 11: f(x) = 4.08839843, alpha = 1.0000
Iter 12: f(x) = 4.08519140, alpha = 1.0000
Iter 13: f(x) = 4.08198333, alpha = 1.0000
Iter 14: f(x) = 4.07877421, alpha = 1.0000
Iter 15: f(x) = 4.07556404, alpha = 1.0000
Iter 16: f(x) = 4.07235282, alpha = 1.0000
Iter 17: f(x) = 4.06914056, alpha = 1.0000
Iter 18: f(x) = 4.06592724, alpha = 1.0000
Iter 19: f(x) = 4.06271286, alpha = 1.0000
Iter 20: f(x) = 4.05949744, alpha = 1.0000
Iter 21: f(x) = 4.05628095, alpha = 1.0000
Iter 22: f(x) = 4.05306341, alpha = 1.0000
Iter 23: f(x) = 4.049